# Full DPO Fine-tuning for Wan2.2 T2V Model

This notebook implements **Direct Preference Optimization (DPO)** directly on the Wan2.2 video generation model.

## What This Does

- ✅ **Optimizes Wan2.2 T2V model directly** (not just a reward model)
- ✅ **Uses preference pairs** to improve video generation
- ✅ **Follows CS234 DPO pattern** adapted for diffusion models
- ✅ **Output**: Fine-tuned video generator that produces higher-quality videos

## Key Differences from Reward Model Training

| Aspect | Reward Model | DPO Fine-tuning (This) |
|--------|--------------|------------------------|
| What it trains | Preference classifier | Wan2.2 T2V model |
| Modifies T2V? | ❌ No | ✅ Yes |
| Output | Evaluation model | Better video generator |
| Time | ~30 min | ~2-4 hours |
| Memory | ~8GB | ~24GB+ |

---

## 🔧 DPO Formula (CS234 Reference)

From CS234 `run_dpo.py:268-269`:
```python
logits = beta * ((log_pi_w - log_ref_w) - (log_pi_l - log_ref_l))
loss = -logsigmoid(logits)
```

For diffusion models, we use **negative denoising error** as implicit reward:
```python
reward = -mse(predicted_noise, true_noise)
```

Lower denoising error = higher quality = higher reward!

## 0. Setup & Installation

In [ ]:
# Install required packages
!pip install -q git+https://github.com/huggingface/diffusers  # Wan2.2 requires latest diffusers
!pip install -q torch torchvision transformers accelerate
!pip install -q opencv-python-headless pillow imageio
!pip install -q numpy matplotlib scipy tqdm
!pip install -q bitsandbytes  # For 8-bit optimizer (memory efficiency)

In [ ]:
import copy
import json
import os
import time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from diffusers import WanPipeline, AutoencoderKLWan, DDPMScheduler
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Configuration

In [ ]:
class Config:
    # Model settings
    wan22_model = "Wan-AI/Wan2.2-T2V-A14B-Diffusers"  # Diffusers-compatible version
    device = device

    # Data paths
    pref_data_path = "video_rankings3_pairwise.json"
    videos_dir = "./wan22-dataset/videos"
    output_dir = "wan22_dpo_finetuned"

    # DPO hyperparameters
    beta = 0.1              # DPO temperature (controls preference signal strength)
    lr = 1e-6               # Learning rate (diffusion models are sensitive!)
    batch_size = 1          # Start small due to memory
    gradient_accumulation_steps = 4  # Effective batch size = 4
    num_epochs = 10

    # Diffusion settings
    n_frames = 8            # Frames per video (reduce for memory)
    num_inference_steps = 40
    guidance_scale = 4.0

    # Training settings
    max_grad_norm = 1.0
    save_every = 500        # Save checkpoint every N steps
    eval_every = 100
    seed = 42

    # Create output directory
    os.makedirs(output_dir, exist_ok=True)

cfg = Config()

# Set seeds
torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)

print("Configuration:")
print(f"  Model: {cfg.wan22_model}")
print(f"  Beta: {cfg.beta}")
print(f"  Learning rate: {cfg.lr}")
print(f"  Batch size: {cfg.batch_size} (effective: {cfg.batch_size * cfg.gradient_accumulation_steps})")
print(f"  Epochs: {cfg.num_epochs}")
print(f"  Frames per video: {cfg.n_frames}")
print(f"  Output: {cfg.output_dir}")

## 2. Download Videos & Preference Data

In [ ]:
# Upload preference data if not found
if not os.path.exists(cfg.pref_data_path):
    try:
        from google.colab import files
        print("Upload video_rankings3_pairwise.json:")
        uploaded = files.upload()
        cfg.pref_data_path = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{cfg.pref_data_path} not found.")

print(f"Preference data: {cfg.pref_data_path}")

In [ ]:
from huggingface_hub import snapshot_download
import glob

if not os.path.isdir(cfg.videos_dir):
    print(f"Downloading videos (~1.2 GB) ...")
    snapshot_download(
        "hivamoh/wan22-physics-videos",
        local_dir="./wan22-dataset",
        allow_patterns=["videos/*.mp4"],
        repo_type="dataset",
    )
    print("Download complete.")
else:
    print(f"Videos already downloaded at {cfg.videos_dir}")

n_videos = len(glob.glob(os.path.join(cfg.videos_dir, "*.mp4")))
print(f"Found {n_videos} video files.")

## 3. Dataset

In [ ]:
class PreferencePairDataset(Dataset):
    """Dataset for DPO training on video preference pairs."""

    def __init__(self, pref_data_path, videos_dir):
        with open(pref_data_path) as f:
            raw_data = json.load(f)

        self.pairs = []
        self.prompts = []

        for group_data in raw_data.values():
            prompt = group_data["prompt"]
            for pair in group_data["pairwise_comparisons"]:
                if pair.get("tie"):
                    continue

                preferred_path = os.path.join(videos_dir, pair["preferred"])
                rejected_path = os.path.join(videos_dir, pair["rejected"])

                if os.path.exists(preferred_path) and os.path.exists(rejected_path):
                    self.pairs.append({
                        "prompt": prompt,
                        "preferred_video": preferred_path,
                        "rejected_video": rejected_path,
                    })
                    self.prompts.append(prompt)

        print(f"Loaded {len(self.pairs)} preference pairs")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        return self.pairs[idx]


def collate_fn(batch):
    """Collate function for DataLoader."""
    return {
        "prompts": [item["prompt"] for item in batch],
        "preferred_videos": [item["preferred_video"] for item in batch],
        "rejected_videos": [item["rejected_video"] for item in batch],
    }


# Load dataset
print("Loading preference dataset...")
dataset = PreferencePairDataset(cfg.pref_data_path, cfg.videos_dir)
dataloader = DataLoader(
    dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)
print(f"Created DataLoader with {len(dataset)} samples")

## 4. Video Loading Utilities

In [ ]:
def load_video_to_latent(video_path, vae, n_frames=16, device="cuda"):
    """Load video and encode to VAE latent space."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Sample frames uniformly
    if total_frames < n_frames:
        indices = list(range(total_frames))
    else:
        indices = np.linspace(0, total_frames - 1, n_frames, dtype=int)

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (256, 256))
            frames.append(frame)
    cap.release()

    # Pad if needed
    while len(frames) < n_frames:
        frames.append(frames[-1] if frames else np.zeros((256, 256, 3), dtype=np.uint8))

    # Convert to tensor and normalize
    video_array = np.stack(frames)
    video_tensor = torch.from_numpy(video_array).permute(0, 3, 1, 2).float()
    video_tensor = (video_tensor / 127.5) - 1.0  # [-1, 1]

    # Encode to latent space
    video_tensor = video_tensor.to(device)
    with torch.no_grad():
        latents = []
        for frame in video_tensor:
            latent_dist = vae.encode(frame.unsqueeze(0)).latent_dist
            latent = latent_dist.sample()
            latents.append(latent)
        latents = torch.cat(latents, dim=0)

    return latents

print("Video loading utilities defined.")

## 5. DPO Loss Function

In [ ]:
def compute_dpo_loss(
    policy_transformer,
    reference_transformer,
    vae,
    text_encoder,
    tokenizer,
    noise_scheduler,
    batch,
    cfg,
):
    """
    Compute DPO loss for Wan2.2 diffusion transformer.

    Following CS234 DPO (run_dpo.py:268-269):
        logits = beta * ((log_pi_w - log_ref_w) - (log_pi_l - log_ref_l))
        loss = -logsigmoid(logits)

    For diffusion models, we use negative denoising error as implicit log prob:
        reward = -mse(predicted_noise, true_noise)
    """
    device = cfg.device

    # Encode text prompts
    text_inputs = tokenizer(
        batch["prompts"],
        padding="max_length",
        max_length=tokenizer.model_max_length,
        truncation=True,
        return_tensors="pt",
    )
    text_embeddings = text_encoder(text_inputs.input_ids.to(device))[0]

    # Load and encode videos to latent space
    preferred_latents = []
    rejected_latents = []

    for pref_path, rej_path in zip(batch["preferred_videos"], batch["rejected_videos"]):
        pref_latent = load_video_to_latent(pref_path, vae, cfg.n_frames, device)
        rej_latent = load_video_to_latent(rej_path, vae, cfg.n_frames, device)
        preferred_latents.append(pref_latent)
        rejected_latents.append(rej_latent)

    preferred_latents = torch.stack(preferred_latents)  # (B, T, C, H, W)
    rejected_latents = torch.stack(rejected_latents)

    # Sample random timestep for each video
    bsz = preferred_latents.shape[0]
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device
    ).long()

    # Add noise to latents (diffusion forward process)
    noise_w = torch.randn_like(preferred_latents)
    noise_l = torch.randn_like(rejected_latents)

    noisy_latents_w = noise_scheduler.add_noise(preferred_latents, noise_w, timesteps)
    noisy_latents_l = noise_scheduler.add_noise(rejected_latents, noise_l, timesteps)

    # Predict noise with policy transformer
    # For video: we process frame by frame
    policy_pred_w = []
    policy_pred_l = []

    for t in range(cfg.n_frames):
        # Preferred
        pred_w = policy_transformer(
            noisy_latents_w[:, t],
            timesteps,
            encoder_hidden_states=text_embeddings,
        ).sample
        policy_pred_w.append(pred_w)

        # Rejected
        pred_l = policy_transformer(
            noisy_latents_l[:, t],
            timesteps,
            encoder_hidden_states=text_embeddings,
        ).sample
        policy_pred_l.append(pred_l)

    policy_pred_w = torch.stack(policy_pred_w, dim=1)
    policy_pred_l = torch.stack(policy_pred_l, dim=1)

    # Predict noise with reference transformer (frozen)
    with torch.no_grad():
        ref_pred_w = []
        ref_pred_l = []

        for t in range(cfg.n_frames):
            # Preferred
            pred_w = reference_transformer(
                noisy_latents_w[:, t],
                timesteps,
                encoder_hidden_states=text_embeddings,
            ).sample
            ref_pred_w.append(pred_w)

            # Rejected
            pred_l = reference_transformer(
                noisy_latents_l[:, t],
                timesteps,
                encoder_hidden_states=text_embeddings,
            ).sample
            ref_pred_l.append(pred_l)

        ref_pred_w = torch.stack(ref_pred_w, dim=1)
        ref_pred_l = torch.stack(ref_pred_l, dim=1)

    # Compute implicit rewards (negative MSE)
    # Lower denoising error = higher reward
    policy_reward_w = -F.mse_loss(policy_pred_w, noise_w, reduction="none").mean(dim=[1, 2, 3, 4])
    policy_reward_l = -F.mse_loss(policy_pred_l, noise_l, reduction="none").mean(dim=[1, 2, 3, 4])
    ref_reward_w = -F.mse_loss(ref_pred_w, noise_w, reduction="none").mean(dim=[1, 2, 3, 4])
    ref_reward_l = -F.mse_loss(ref_pred_l, noise_l, reduction="none").mean(dim=[1, 2, 3, 4])

    # DPO loss (same structure as CS234)
    logits = cfg.beta * ((policy_reward_w - ref_reward_w) - (policy_reward_l - ref_reward_l))
    loss = -F.logsigmoid(logits).mean()

    # Compute accuracy (how often policy prefers preferred > rejected)
    with torch.no_grad():
        accuracy = (logits > 0).float().mean().item()
        reward_margin = logits.mean().item()

    metrics = {
        "loss": loss.item(),
        "accuracy": accuracy,
        "reward_margin": reward_margin,
    }

    return loss, metrics

print("DPO loss function defined.")

## 6. Load Wan2.2 Model

In [ ]:
print("=" * 70)
print("  Loading Wan2.2 T2V Model")
print("=" * 70)
print(f"Model: {cfg.wan22_model}")
print()

# Load VAE separately (float32 for stability)
print("Loading VAE...")
vae = AutoencoderKLWan.from_pretrained(
    cfg.wan22_model,
    subfolder="vae",
    torch_dtype=torch.float32
)
print(f"✓ Loaded Wan2.2 VAE (compression: 4×16×16)")

# Load full pipeline
print("\nLoading full pipeline...")
pipe = WanPipeline.from_pretrained(
    cfg.wan22_model,
    vae=vae,
    torch_dtype=torch.bfloat16,  # Use bfloat16 for MoE model
)
pipe.to(cfg.device)
print("✓ Pipeline loaded successfully")

# Extract components (Wan2.2 uses transformer, not unet)
policy_transformer = pipe.transformer
vae = pipe.vae
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)

print(f"\n✓ Extracted components:")
print(f"  - Transformer: {type(policy_transformer).__name__}")
print(f"  - VAE: {type(vae).__name__}")
print(f"  - Text Encoder: {type(text_encoder).__name__}")

## 7. Create Reference Model & Setup Training

In [ ]:
# Create frozen reference copy of transformer
print("Creating reference transformer (frozen)...")
reference_transformer = copy.deepcopy(policy_transformer)
reference_transformer.requires_grad_(False)
reference_transformer.eval()
print("✓ Reference transformer created and frozen")

# Freeze VAE and text encoder
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
vae.eval()
text_encoder.eval()
print("✓ VAE and text encoder frozen")

# Only train policy transformer
policy_transformer.train()
policy_transformer.requires_grad_(True)
print("✓ Policy transformer set to trainable")

# Count parameters
trainable_params = sum(p.numel() for p in policy_transformer.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in policy_transformer.parameters())
print(f"\nTrainable parameters: {trainable_params:,} / {total_params:,}")

In [ ]:
# Optimizer (use AdamW8bit for memory efficiency if available)
print("\nSetting up optimizer...")
try:
    import bitsandbytes as bnb
    optimizer = bnb.optim.AdamW8bit(
        policy_transformer.parameters(),
        lr=cfg.lr,
        betas=(0.9, 0.999),
        weight_decay=0.01,
    )
    print("✓ Using 8-bit AdamW optimizer (memory efficient)")
except ImportError:
    optimizer = torch.optim.AdamW(
        policy_transformer.parameters(),
        lr=cfg.lr,
        betas=(0.9, 0.999),
        weight_decay=0.01,
    )
    print("✓ Using standard AdamW optimizer")

print(f"  Learning rate: {cfg.lr}")
print(f"  Weight decay: 0.01")

## 8. Training Loop

In [ ]:
print("\n" + "=" * 70)
print("  Starting DPO Training")
print("=" * 70)
print(f"Beta: {cfg.beta}")
print(f"Epochs: {cfg.num_epochs}")
print(f"Batch size: {cfg.batch_size} (effective: {cfg.batch_size * cfg.gradient_accumulation_steps})")
print(f"Training samples: {len(dataset)}")
print("=" * 70)
print()

# Training tracking
global_step = 0
best_accuracy = 0.0
training_history = {
    "loss": [],
    "accuracy": [],
    "reward_margin": [],
}

# Training loop
for epoch in range(cfg.num_epochs):
    epoch_metrics = defaultdict(list)

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{cfg.num_epochs}")

    for step, batch in enumerate(progress_bar):
        # Compute DPO loss
        loss, metrics = compute_dpo_loss(
            policy_transformer,
            reference_transformer,
            vae,
            text_encoder,
            tokenizer,
            noise_scheduler,
            batch,
            cfg,
        )

        # Backward pass
        loss = loss / cfg.gradient_accumulation_steps
        loss.backward()

        # Update weights
        if (step + 1) % cfg.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(policy_transformer.parameters(), cfg.max_grad_norm)
            optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        # Log metrics
        for k, v in metrics.items():
            epoch_metrics[k].append(v)

        progress_bar.set_postfix({
            "loss": f"{metrics['loss']:.4f}",
            "acc": f"{metrics['accuracy']:.3f}",
            "margin": f"{metrics['reward_margin']:.3f}",
        })

        # Save checkpoint
        if global_step > 0 and global_step % cfg.save_every == 0:
            save_path = os.path.join(cfg.output_dir, f"checkpoint-{global_step}")
            policy_transformer.save_pretrained(save_path)
            print(f"\nSaved checkpoint to {save_path}")

    # Epoch summary
    avg_loss = np.mean(epoch_metrics["loss"])
    avg_acc = np.mean(epoch_metrics["accuracy"])
    avg_margin = np.mean(epoch_metrics["reward_margin"])

    # Record history
    training_history["loss"].append(avg_loss)
    training_history["accuracy"].append(avg_acc)
    training_history["reward_margin"].append(avg_margin)

    print(f"\nEpoch {epoch+1} Summary:")
    print(f"  Loss: {avg_loss:.4f}")
    print(f"  Accuracy: {avg_acc:.3f}")
    print(f"  Reward Margin: {avg_margin:.3f}")

    # Save best model
    if avg_acc > best_accuracy:
        best_accuracy = avg_acc
        save_path = os.path.join(cfg.output_dir, "best_model")
        policy_transformer.save_pretrained(save_path)
        print(f"  ✓ New best model saved (accuracy: {avg_acc:.3f})")

    print()

# Final save
final_path = os.path.join(cfg.output_dir, "final_model")
policy_transformer.save_pretrained(final_path)
print("\n" + "=" * 70)
print(f"Training complete!")
print(f"Best accuracy: {best_accuracy:.3f}")
print(f"Final model saved to: {final_path}")
print(f"Best model saved to: {os.path.join(cfg.output_dir, 'best_model')}")
print("=" * 70)

## 9. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(training_history["loss"], marker='o')
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("DPO Loss")
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(training_history["accuracy"], marker='o', color='green')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Preference Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Reward margin
axes[2].plot(training_history["reward_margin"], marker='o', color='orange')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Reward Margin")
axes[2].set_title("Reward Margin")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(cfg.output_dir, "training_curves.png"), dpi=150)
plt.show()

print(f"Training curves saved to {cfg.output_dir}/training_curves.png")

## 10. Generate Video with Fine-tuned Model

Test your fine-tuned model!

In [ ]:
# Load fine-tuned model into pipeline
print("Loading fine-tuned transformer...")
from diffusers import WanTransformer2DModel

finetuned_transformer = WanTransformer2DModel.from_pretrained(
    os.path.join(cfg.output_dir, "best_model"),
    torch_dtype=torch.bfloat16,
)

# Create new pipeline with fine-tuned model
pipe_finetuned = WanPipeline.from_pretrained(
    cfg.wan22_model,
    transformer=finetuned_transformer,
    torch_dtype=torch.bfloat16,
)
pipe_finetuned.to(cfg.device)

print("✓ Fine-tuned model loaded!")

In [ ]:
# Generate a test video
test_prompt = "A ball rolling down a slope with realistic physics"

print(f"Generating video for: '{test_prompt}'")
print("This may take a few minutes...")

output = pipe_finetuned(
    prompt=test_prompt,
    height=720,
    width=1280,
    num_frames=81,
    guidance_scale=4.0,
    num_inference_steps=40,
)

frames = output.frames[0]
print(f"\n✓ Generated {len(frames)} frames at 720P")

# Save as video
import imageio
output_path = os.path.join(cfg.output_dir, "generated_video.mp4")
imageio.mimsave(output_path, [np.array(f) for f in frames], fps=24)
print(f"✓ Saved video to: {output_path}")

## Summary

✅ **Training Complete!**

Your Wan2.2 T2V model has been fine-tuned using DPO!

### What You Now Have:
- Fine-tuned transformer that prefers higher-quality videos
- Training checkpoints saved to `{output_dir}/`
- Best model at `{output_dir}/best_model`
- Training curves visualization

### Expected Results:
- **Preference Accuracy**: 70-80% (policy prefers preferred > rejected)
- **Generated Videos**: Better physics, clearer motion, higher fidelity

### Next Steps:
1. Generate more videos with different prompts
2. Compare base model vs fine-tuned model outputs
3. Iterate: collect more preferences, retrain

### Files:
- **Best model**: `{output_dir}/best_model`
- **Training curves**: `{output_dir}/training_curves.png`
- **Checkpoints**: `{output_dir}/checkpoint-*`